# PyTorch interview coding lab

Install compatible PyTorch and NumPy builds into the notebook kernel environment if missing. Exercises use CPU tensors; no CUDA device or model download is needed.

Use only the synthetic fixtures supplied here. No private data is included. This notebook never installs packages automatically. Replace each `...`, run the exercise cell, then its separate test cell. Failures before completion are intentional. Tests cover selected cases, not every production input. Reference solutions are text at the end and are not executed automatically.

Original exercises cover topics from the [Devinterview PyTorch checklist](https://github.com/Devinterview-io/pytorch-interview-questions). Its answers are not reproduced. Existing exercise IDs and the first six exercises are unchanged.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('Exercise device: CPU; no CUDA execution required')


## 01 / TENSORS & SHAPES: Split tokens into attention heads

Transform [batch, tokens, width] into [batch, heads, tokens, head_width] without scrambling token order.

### Hints
- **Reshape into [B, T, H, D]:** Keep batch and token dimensions first, then split width into heads and per-head width.
- **Move heads before tokens:** Swap axes 1 and 2. A transpose is often a non-contiguous view.

### Expected behavior
For x shaped [2, 4, 12] and heads=3: output shape [2, 3, 4, 4]. Head slices preserve the original token's contiguous width groups.


In [ ]:
import torch

def split_heads(x: torch.Tensor, heads: int) -> torch.Tensor:
    batch, tokens, width = x.shape
    if heads <= 0 or width % heads:
        raise ValueError("heads must divide width")
    separated = ...
    return ...


In [ ]:
x = torch.arange(96.0).reshape(2, 4, 12)
y = split_heads(x, 3)
assert y.shape == (2, 3, 4, 4)
assert torch.equal(y[0, 1, 2], x[0, 2, 4:8])
restored = y.transpose(1, 2).reshape(2, 4, 12)
assert torch.equal(restored, x)
try:
    split_heads(x, 5)
except ValueError:
    pass
else:
    raise AssertionError("Invalid head count must be rejected")
print("PASS: head shape, token ordering, reconstruction, invalid heads")


## 02 / NUMERICAL STABILITY: Write numerically stable softmax

Normalize the last dimension without overflow, preserving batch dimensions and gradient flow.

### Hints
- **Subtract the row maximum:** A per-row maximum with keepdim=True broadcasts over the original last axis.
- **Exponentiate shifted logits:** Subtracting a constant leaves softmax unchanged but prevents exp(large positive number).
- **Sum weights along the last axis:** Do not sum across the batch. Preserve the final axis for broadcasting.

### Expected behavior
For [1000, 1001, 1002], probabilities are approximately [0.0900, 0.2447, 0.6652], all finite, with row sum 1.


In [ ]:
import torch

def stable_probs(logits: torch.Tensor) -> torch.Tensor:
    shifted = ...
    weights = ...
    return weights / ...


In [ ]:
logits = torch.tensor([[1000., 1001., 1002.], [3., 1., -2.]], requires_grad=True)
probs = stable_probs(logits)
assert torch.isfinite(probs).all()
assert torch.allclose(probs.sum(dim=-1), torch.ones(2))
assert torch.allclose(probs, torch.softmax(logits, dim=-1), atol=1e-6)
probs[:, 0].sum().backward()
assert logits.grad is not None and torch.isfinite(logits.grad).all()
print("PASS: large logits, normalization, reference parity, gradients")


## 03 / PYTORCH SDPA: Make future tokens invisible

Create a boolean causal mask with the correct SDPA semantics and explicitly disable functional dropout.

### Hints
- **Construct the allowed-position mask:** A lower triangular boolean matrix includes current and previous keys. In SDPA, True means allowed.
- **Disable SDPA dropout for this inference exercise:** Functional SDPA applies the provided dropout probability regardless of an enclosing module's eval setting.

### Expected behavior
Output has the same [B,H,T,D] shape as Q. Perturbing K and V at the final position must not change earlier outputs.


In [ ]:
import torch
from torch.nn import functional as F

def causal_attention(q: torch.Tensor, k: torch.Tensor,
                     v: torch.Tensor) -> torch.Tensor:
    tokens = q.size(-2)
    if k.size(-2) != tokens or v.size(-2) != tokens:
        raise ValueError("This exercise requires equal sequence lengths")
    allowed = ...
    return F.scaled_dot_product_attention(
        q, k, v, attn_mask=allowed, dropout_p=...
    )


In [ ]:
torch.manual_seed(7)
q, k, v = [torch.randn(2, 2, 4, 8) for _ in range(3)]
out = causal_attention(q, k, v)
assert out.shape == q.shape
changed_k, changed_v = k.clone(), v.clone()
changed_k[..., -1, :] += 100
changed_v[..., -1, :] -= 100
changed = causal_attention(q, changed_k, changed_v)
assert torch.allclose(out[..., :-1, :], changed[..., :-1, :], atol=1e-5)
mask = torch.ones(4, 4, dtype=torch.bool).tril()
scores = (q @ k.transpose(-2, -1)) / (q.size(-1) ** 0.5)
reference = scores.masked_fill(~mask, float('-inf')).softmax(-1) @ v
assert torch.allclose(out, reference, atol=1e-5)
print("PASS: output shape, future-token isolation, explicit attention parity")


## 04 / CAUSAL LANGUAGE-MODEL LOSS: Align next-token targets and mask padding

Write a custom next-token loss that shifts once, ignores -100 labels, and rejects an all-ignored batch.

### Hints
- **Keep predictions before the final position:** The prediction at position t supervises the token at position t+1.
- **Keep targets after the first position:** Pair logits at 0..T-2 with labels at 1..T-1.
- **Specify the ignored label:** Cross entropy must use the same ignore index as your label mask.

### Expected behavior
With zero logits over a 3-token vocabulary, loss is log(3) ≈ 1.0986 over valid targets. Masked targets and unused final-position logits have zero gradient.


In [ ]:
import torch
from torch.nn import functional as F

def shifted_lm_loss(logits: torch.Tensor,
                    labels: torch.Tensor) -> torch.Tensor:
    shifted_logits = ...
    shifted_labels = ...
    if not (shifted_labels != -100).any():
        raise ValueError("At least one supervised target is required")
    return F.cross_entropy(
        shifted_logits.reshape(-1, shifted_logits.size(-1)),
        shifted_labels.reshape(-1), ignore_index=...
    )


In [ ]:
import math
logits = torch.zeros(1, 4, 3, requires_grad=True)
labels = torch.tensor([[0, 1, -100, 2]])
loss = shifted_lm_loss(logits, labels)
assert abs(loss.item() - math.log(3)) < 1e-6
loss.backward()
assert torch.equal(logits.grad[:, 1, :], torch.zeros(1, 3))
assert torch.equal(logits.grad[:, -1, :], torch.zeros(1, 3))
assert logits.grad[:, 0, :].abs().sum() > 0
try:
    shifted_lm_loss(logits.detach(), torch.full_like(labels, -100))
except ValueError:
    pass
else:
    raise AssertionError("All-ignored labels must be rejected")
print("PASS: shifted loss, ignored gradients, last-position gradient, empty supervision")


## 05 / AUTOGRAD & OPTIMIZATION: Wire a clean optimizer update

Clear stale gradients, backpropagate, and update parameters exactly once per training step.

### Hints
- **Clear previous gradients:** backward accumulates. This exercise is one update per batch, not intentional gradient accumulation.
- **Compute gradients:** Backward traverses the autograd graph, filling parameter gradients.
- **Apply the optimizer update:** Computing gradients alone does not update weights.

### Expected behavior
For y = w*x, initial w=0, x=1, target=2 and SGD lr=0.1: first w=0.4; after a second clean update w=0.72.


In [ ]:
import torch
from torch import nn

def train_once(model: nn.Module, optimizer: torch.optim.Optimizer,
               x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    model.train()
    ...
    loss = (model(x) - target).square().mean()
    ...
    ...
    return loss.detach()


In [ ]:
model = nn.Linear(1, 1, bias=False)
with torch.no_grad():
    model.weight.zero_()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
x, target = torch.tensor([[1.]]), torch.tensor([[2.]])
first_loss = train_once(model, optimizer, x, target)
assert abs(first_loss.item() - 4.0) < 1e-6
assert abs(model.weight.item() - 0.4) < 1e-6
assert not first_loss.requires_grad
train_once(model, optimizer, x, target)
assert abs(model.weight.item() - 0.72) < 1e-6
print("PASS: backward, parameter update, gradient clearing, detached logging loss")


## 06 / INFERENCE CORRECTNESS: Separate eval mode from autograd

Disable training-only module behavior and gradient recording independently.

### Hints
- **Switch module behavior:** Dropout changes behavior with the module's training flag, not just with gradient tracking.
- **Enter a no-gradient context:** eval alone does not disable autograd. This prediction is not being reused for training.

### Expected behavior
Repeated predictions are identical with dropout disabled, probabilities sum to one, and returned tensors do not require gradients.


In [ ]:
import torch
from torch import nn

def predict(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    ...
    with ...:
        return torch.softmax(model(x), dim=-1)


In [ ]:
torch.manual_seed(11)
model = nn.Sequential(nn.Linear(4, 3), nn.Dropout(p=0.9))
model.train()
x = torch.randn(2, 4, requires_grad=True)
a, b = predict(model, x), predict(model, x)
assert not model.training
assert not a.requires_grad
assert torch.equal(a, b)
assert torch.allclose(a.sum(-1), torch.ones(2))
assert x.grad is None
print("PASS: eval behavior, no autograd, deterministic dropout, normalized output")


## 07 / NUMPY & TENSOR OWNERSHIP: Share NumPy storage, then make an independent copy

Distinguish a CPU tensor that aliases a writable NumPy array from a cloned tensor with independent storage.

### Hints
- **Create a CPU tensor sharing the NumPy array:** For this writable float32 CPU array, from_numpy shares memory rather than copying.
- **Make independent tensor storage:** detach breaks autograd history but still shares storage. clone is the copy operation needed here.

### Expected behavior
For [2, 4, 8], setting shared[1]=9 also changes array[1], but independent[1] remains 4. All tensors are on CPU with dtype float32.


In [ ]:
import numpy as np
import torch

def bridge_array(array: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    if not array.flags.writeable:
        raise ValueError("Use a writable input array")
    shared = ...
    independent = ...
    return shared, independent


In [ ]:
array = np.array([2., 4., 8.], dtype=np.float32)
shared, independent = bridge_array(array)
assert shared.dtype == independent.dtype == torch.float32
assert shared.device.type == 'cpu'
assert shared.data_ptr() != independent.data_ptr()
shared[1] = 9
assert array[1] == 9 and independent[1].item() == 4
array[2] = 12
assert shared[2].item() == 12 and independent[2].item() == 8
independent[0] = -3
assert array[0] == 2
readonly = np.array([1.], dtype=np.float32)
readonly.flags.writeable = False
try:
    bridge_array(readonly)
except ValueError:
    pass
else:
    raise AssertionError('Read-only input must be rejected')
print('PASS: shared storage, independent clone, dtype, read-only rejection')


## 08 / LEAF & NON-LEAF GRADIENTS: Inspect gradients through a changing computation graph

Retain an intermediate gradient, choose a forward branch, and verify the chain rule without accidentally accumulating old leaf gradients.

### Hints
- **Keep the non-leaf h gradient for inspection:** Autograd computes intermediate derivatives, but non-leaf .grad is not normally populated unless retained.
- **Use squared or linear loss according to the branch:** The dynamic forward graph includes only the selected branch. Sum produces a scalar loss.
- **Compute gradients:** The chain rule carries dloss/dh through h=3*x to x. It does not update x.

### Expected behavior
At x=2: h=6. Squared branch: loss=36, h.grad=12, x.grad=36. Linear branch: loss=6, h.grad=1, x.grad=3.


In [ ]:
import torch

def branch_grad(x: torch.Tensor, squared: bool) -> tuple[torch.Tensor, torch.Tensor]:
    if not x.is_leaf or not x.requires_grad:
        raise ValueError("x must be a leaf requiring gradients")
    x.grad = None
    h = 3 * x
    ...
    loss = ...
    ...
    return h, loss.detach()


In [ ]:
x = torch.tensor([2.], requires_grad=True)
h, loss = branch_grad(x, True)
assert h.grad is not None and x.grad is not None
assert h.item() == 6 and loss.item() == 36
assert h.grad.item() == 12 and x.grad.item() == 36
h2, loss2 = branch_grad(x, False)
assert h2.grad is not None
assert loss2.item() == 6 and h2.grad.item() == 1 and x.grad.item() == 3
assert not loss2.requires_grad
negative = torch.tensor([-1.], requires_grad=True)
branch_grad(negative, True)
assert negative.grad.item() == -18
try:
    branch_grad(torch.tensor([2.]), True)
except ValueError:
    pass
else:
    raise AssertionError('A leaf requiring gradients is required')
print('PASS: branch-specific chain rule, retained non-leaf gradient, fresh gradients')


## 09 / NN.MODULE & PARAMETERS: Register a custom layer's learnable and fixed state

Build a composable feature-wise scale layer with a learned parameter, a non-learned buffer, and a differentiable forward method.

### Hints
- **Register a trainable feature scale:** Assigning an nn.Parameter to a Module attribute makes it discoverable by model.parameters().
- **Register a fixed offset buffer:** A persistent buffer moves with the module and is in state_dict, but is not an optimizer parameter.
- **Compute the feature-wise affine result:** The [width] vectors broadcast over leading batch dimensions. Implement forward, not __call__.

### Expected behavior
At initialization the layer preserves its input. After summing output [1,2,3], scale.grad=[1,2,3]; SGD at 0.1 changes scale to [0.9,0.8,0.7] while offset stays zero.


In [ ]:
import torch
from torch import nn

class FeatureScale(nn.Module):
    def __init__(self, width: int) -> None:
        super().__init__()
        if width <= 0:
            raise ValueError("width must be positive")
        self.scale = ...
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] != self.scale.numel():
            raise ValueError("Input feature width does not match")
        return ...


In [ ]:
layer = FeatureScale(3)
x = torch.tensor([[1., 2., 3.]])
assert torch.equal(layer(x), x)
assert list(dict(layer.named_parameters())) == ['scale']
assert list(dict(layer.named_buffers())) == ['offset']
assert set(layer.state_dict()) == {'scale', 'offset'}
optimizer = torch.optim.SGD(layer.parameters(), lr=0.1)
layer(x).sum().backward()
assert torch.equal(layer.scale.grad, torch.tensor([1., 2., 3.]))
optimizer.step()
assert torch.allclose(layer.scale, torch.tensor([0.9, 0.8, 0.7]))
assert torch.equal(layer.offset, torch.zeros(3))
assert nn.Sequential(layer, nn.ReLU())(x).shape == x.shape
layer.to(dtype=torch.float64)
assert layer.scale.dtype == layer.offset.dtype == torch.float64
try:
    layer(torch.ones(2, 4))
except ValueError:
    pass
else:
    raise AssertionError('Mismatched width must fail')
print('PASS: registration, buffers, gradients, optimizer, composition, dtype move')


## 10 / DEVICES & DTYPES: Move a batch without breaking feature and target dtypes

Move features and class indices onto an explicitly chosen device, preserving the different dtypes required by a classification loss.

### Hints
- **Move features as float32:** Tensor.to returns a tensor; it does not necessarily mutate the original tensor or force a copy.
- **Move integer class targets as long:** Index targets for cross-entropy use long, not floating feature dtype. Inputs here already contain integer class IDs.

### Expected behavior
CPU fixture: int16 features become float32; int16 class IDs become long. Shapes and values stay unchanged, and the original tensors keep their dtypes.


In [ ]:
import torch

def prepare_batch(features: torch.Tensor, labels: torch.Tensor,
                  device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    if features.ndim != 2 or labels.ndim != 1 or features.shape[0] != labels.shape[0]:
        raise ValueError("Expected [batch, features] and [batch] labels")
    x = ...
    y = ...
    return x, y


In [ ]:
features = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.int16)
labels = torch.tensor([0, 2], dtype=torch.int16)
x, y = prepare_batch(features, labels, torch.device('cpu'))
assert x.dtype == torch.float32 and y.dtype == torch.long
assert x.device.type == y.device.type == 'cpu'
assert torch.equal(x, features.float()) and torch.equal(y, labels.long())
assert features.dtype == labels.dtype == torch.int16
loss = torch.nn.functional.cross_entropy(x, y)
assert torch.isfinite(loss)
try:
    prepare_batch(features, labels[:1], torch.device('cpu'))
except ValueError:
    pass
else:
    raise AssertionError('Mismatched batch sizes must fail')
print('PASS: CPU transfer, feature/label dtypes, preserved values, shape rejection')


## 11 / LEARNING-RATE SCHEDULING: Apply the learning rate to the intended optimizer update

Record the rate actually used for each update and advance a StepLR only after optimizer.step in this per-update schedule.

### Hints
- **Read the optimizer's current rate before the update:** This records the rate actually used by the following optimizer.step.
- **Update model weights first:** Backward only computes gradients; optimizer.step applies the current rate.
- **Advance this per-update schedule afterward:** With step_size=2, the next pair of updates uses half the previous pair's rate.

### Expected behavior
Four optimizer updates use [0.4,0.4,0.2,0.2], and the scalar weight ends near 0.9856. Six updates use [0.4,0.4,0.2,0.2,0.1,0.1].


In [ ]:
import torch
from torch import nn

def scheduled_updates(steps: int) -> tuple[list[float], torch.Tensor]:
    if steps < 0:
        raise ValueError("steps must be nonnegative")
    model = nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        model.weight.zero_()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
    rates = []
    for _ in range(steps):
        rates.append(...)
        optimizer.zero_grad(set_to_none=True)
        loss = (model(torch.ones(1, 1)) - 1).square().mean()
        loss.backward()
        ...
        ...
    return rates, model.weight.detach().clone()


In [ ]:
rates, weight = scheduled_updates(4)
assert rates == [0.4, 0.4, 0.2, 0.2]
assert abs(weight.item() - 0.9856) < 1e-6
six, _ = scheduled_updates(6)
assert six == [0.4, 0.4, 0.2, 0.2, 0.1, 0.1]
empty, initial = scheduled_updates(0)
assert empty == [] and initial.item() == 0
try:
    scheduled_updates(-1)
except ValueError:
    pass
else:
    raise AssertionError('Negative step count must fail')
print('PASS: rate sequence, update ordering, parameter value, zero/invalid steps')


## 12 / BATCH-WEIGHTED GRADIENTS: Accumulate unequal microbatches with the correct weighting

Match one full-batch mean-squared-error SGD update using unequal microbatches and a single optimizer step.

### Hints
- **Clear gradients once before all microbatches:** Do not clear gradients inside this accumulation window.
- **Normalize each summed error by the full target-element count:** Equal weighting of microbatch means is wrong when their target counts differ.
- **Accumulate this microbatch's gradient:** Each microbatch builds a fresh graph. The single optimizer.step comes after the loop.

### Expected behavior
For x=[1,2,3], target=2*x, initial w=0 and lr=0.1, splitting into sizes 1 and 2 must match the full-batch loss 56/3 and updated weight 28/15.


In [ ]:
import torch
from torch import nn

def accumulated_step(model: nn.Module, optimizer: torch.optim.Optimizer,
                     batches: list[tuple[torch.Tensor, torch.Tensor]]) -> float:
    total_targets = sum(target.numel() for _, target in batches)
    if total_targets == 0:
        raise ValueError("At least one target element is required")
    model.train()
    ...
    logged_loss = 0.0
    for x, target in batches:
        prediction = model(x)
        if prediction.shape != target.shape:
            raise ValueError("Prediction and target shapes must match")
        squared_error = (prediction - target).square().sum()
        loss = ...
        ...
        logged_loss += loss.detach().item()
    optimizer.step()
    return logged_loss


In [ ]:
model = nn.Linear(1, 1, bias=False)
reference = nn.Linear(1, 1, bias=False)
with torch.no_grad():
    model.weight.zero_()
    reference.weight.zero_()
x = torch.tensor([[1.], [2.], [3.]])
target = 2 * x
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
ref_optimizer = torch.optim.SGD(reference.parameters(), lr=0.1)
ref_optimizer.zero_grad(set_to_none=True)
ref_loss = (reference(x) - target).square().mean()
ref_loss.backward()
ref_optimizer.step()
logged = accumulated_step(model, optimizer, [(x[:1], target[:1]), (x[1:], target[1:])])
assert abs(logged - 56 / 3) < 1e-5
assert torch.allclose(model.weight, reference.weight, atol=1e-6)
assert abs(model.weight.item() - 28 / 15) < 1e-6
for invalid in [[], [(x, target.reshape(-1))]]:
    try:
        accumulated_step(model, optimizer, invalid)
    except ValueError:
        pass
    else:
        raise AssertionError('Empty supervision and broadcast-shaped targets must fail')
print('PASS: unequal microbatches, full-batch parity, single update, invalid targets')


# Reference solutions

Try the exercises first. Equivalent implementations can be valid even when the browser's limited reference matcher does not accept them.


## Split tokens into attention heads

```python
import torch

def split_heads(x: torch.Tensor, heads: int) -> torch.Tensor:
    batch, tokens, width = x.shape
    if heads <= 0 or width % heads:
        raise ValueError("heads must divide width")
    separated = x.reshape(batch, tokens, heads, width // heads)
    return separated.transpose(1, 2)
```

Reshaping alone to [B,H,T,D] would group the underlying elements incorrectly. First split the final width axis, then permute axes. reshape also supports inputs whose current strides prevent view.


## Write numerically stable softmax

```python
import torch

def stable_probs(logits: torch.Tensor) -> torch.Tensor:
    shifted = logits - logits.amax(dim=-1, keepdim=True)
    weights = torch.exp(shifted)
    return weights / weights.sum(dim=-1, keepdim=True)
```

Subtracting a shared row constant cancels in softmax's numerator and denominator. Reducing the last axis independently keeps separate batch items from influencing one another. All-masked rows need a separate policy; this exercise assumes finite, nonempty logits.


## Make future tokens invisible

```python
import torch
from torch.nn import functional as F

def causal_attention(q: torch.Tensor, k: torch.Tensor,
                     v: torch.Tensor) -> torch.Tensor:
    tokens = q.size(-2)
    if k.size(-2) != tokens or v.size(-2) != tokens:
        raise ValueError("This exercise requires equal sequence lengths")
    allowed = torch.ones(tokens, tokens, dtype=torch.bool, device=q.device).tril()
    return F.scaled_dot_product_attention(
        q, k, v, attn_mask=allowed, dropout_p=0.0
    )
```

Mask scores before normalization. This exercise is square attention with no padding or KV cache; cached unequal-length attention needs explicit offset handling. We do not pass is_causal together with this explicit mask.


## Align next-token targets and mask padding

```python
import torch
from torch.nn import functional as F

def shifted_lm_loss(logits: torch.Tensor,
                    labels: torch.Tensor) -> torch.Tensor:
    shifted_logits = logits[:, :-1, :]
    shifted_labels = labels[:, 1:]
    if not (shifted_labels != -100).any():
        raise ValueError("At least one supervised target is required")
    return F.cross_entropy(
        shifted_logits.reshape(-1, shifted_logits.size(-1)),
        shifted_labels.reshape(-1), ignore_index=-100
    )
```

This is a custom loss, so it performs its own shift. Many Hugging Face causal-LM classes already shift when labels are supplied—do not apply both. The loss is normalized by valid targets, not all padded positions.


## Wire a clean optimizer update

```python
import torch
from torch import nn

def train_once(model: nn.Module, optimizer: torch.optim.Optimizer,
               x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = (model(x) - target).square().mean()
    loss.backward()
    optimizer.step()
    return loss.detach()
```

The first loss is (0−2)^2=4, giving gradient −4 and update +0.4. The next gradient is −3.2, not −7.2: stale gradients must be cleared. A detached returned loss avoids retaining a training graph in logging history.


## Separate eval mode from autograd

```python
import torch
from torch import nn

def predict(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    model.eval()
    with torch.inference_mode():
        return torch.softmax(model(x), dim=-1)
```

Module mode and autograd mode are separate controls. inference_mode removes extra bookkeeping and imposes additional restrictions; no_grad is also valid for this exercise. Restore train mode before training again. SDPA functional dropout is a separate explicit argument.


## Share NumPy storage, then make an independent copy

```python
import numpy as np
import torch

def bridge_array(array: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    if not array.flags.writeable:
        raise ValueError("Use a writable input array")
    shared = torch.from_numpy(array)
    independent = shared.clone()
    return shared, independent
```

Ownership and gradient tracking are different questions. from_numpy aliases compatible CPU memory; a dtype/device conversion can allocate a new tensor. This exercise assumes a supported writable NumPy dtype and layout; read-only input is explicitly rejected. Do not write through tensors backed by read-only arrays.


## Inspect gradients through a changing computation graph

```python
import torch

def branch_grad(x: torch.Tensor, squared: bool) -> tuple[torch.Tensor, torch.Tensor]:
    if not x.is_leaf or not x.requires_grad:
        raise ValueError("x must be a leaf requiring gradients")
    x.grad = None
    h = 3 * x
    h.retain_grad()
    loss = h.square().sum() if squared else h.sum()
    loss.backward()
    return h, loss.detach()
```

For h=3x and L=h^2, dL/dx=(2h)*3=18x. The linear branch gives 3. Each call rebuilds the graph and deliberately clears x.grad. No retain_graph=True is needed. A detached loss with no remaining differentiable path cannot backward; it does not produce a successful all-zero training step.


## Register a custom layer's learnable and fixed state

```python
import torch
from torch import nn

class FeatureScale(nn.Module):
    def __init__(self, width: int) -> None:
        super().__init__()
        if width <= 0:
            raise ValueError("width must be positive")
        self.scale = nn.Parameter(torch.ones(width))
        self.register_buffer("offset", torch.zeros(width))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] != self.scale.numel():
            raise ValueError("Input feature width does not match")
        return x * self.scale + self.offset
```

Module.__call__ dispatches forward and supports hooks; use model(x) rather than bypassing it. Parameters and buffers have different optimization roles but both participate in state serialization and module device/dtype moves. Sequential can compose this custom layer with other modules.


## Move a batch without breaking feature and target dtypes

```python
import torch

def prepare_batch(features: torch.Tensor, labels: torch.Tensor,
                  device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    if features.ndim != 2 or labels.ndim != 1 or features.shape[0] != labels.shape[0]:
        raise ValueError("Expected [batch, features] and [batch] labels")
    x = features.to(device=device, dtype=torch.float32)
    y = labels.to(device=device, dtype=torch.long)
    return x, y
```

Move the model and all participating tensors to compatible devices. CUDA availability does not prove a speedup, and a CPU test does not validate CUDA kernels. .to may return the same tensor when no conversion is needed. Casting class IDs is not a substitute for validating their range or integral meaning.


## Apply the learning rate to the intended optimizer update

```python
import torch
from torch import nn

def scheduled_updates(steps: int) -> tuple[list[float], torch.Tensor]:
    if steps < 0:
        raise ValueError("steps must be nonnegative")
    model = nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        model.weight.zero_()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
    rates = []
    for _ in range(steps):
        rates.append(optimizer.param_groups[0]["lr"])
        optimizer.zero_grad(set_to_none=True)
        loss = (model(torch.ones(1, 1)) - 1).square().mean()
        loss.backward()
        optimizer.step()
        scheduler.step()
    return rates, model.weight.detach().clone()
```

The scheduling unit is a deliberate choice. Here one scheduler step follows each successful optimizer update; an epoch-based schedule instead advances at epoch boundaries. Metric-driven schedulers have another contract. Mixed-precision skipped updates require matching the schedule to actual updates, not blindly counting backward calls.


## Accumulate unequal microbatches with the correct weighting

```python
import torch
from torch import nn

def accumulated_step(model: nn.Module, optimizer: torch.optim.Optimizer,
                     batches: list[tuple[torch.Tensor, torch.Tensor]]) -> float:
    total_targets = sum(target.numel() for _, target in batches)
    if total_targets == 0:
        raise ValueError("At least one target element is required")
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logged_loss = 0.0
    for x, target in batches:
        prediction = model(x)
        if prediction.shape != target.shape:
            raise ValueError("Prediction and target shapes must match")
        squared_error = (prediction - target).square().sum()
        loss = squared_error / total_targets
        loss.backward()
        logged_loss += loss.detach().item()
    optimizer.step()
    return logged_loss
```

This exercise averages squared error across target elements, not microbatches. The analogous language-model denominator is the total supervised-token count. Exact full-batch equivalence here assumes the deterministic Linear toy, no batch-dependent layers, and no per-microbatch clipping or optimizer updates. Dropout and BatchNorm can change the equivalence assumptions.
